# Config

In [ ]:
%run <Fundraising_Config>

In [ ]:
# Resources
bronze_lakehouse_name = "{BRONZE_LAKEHOUSE_NAME}"

# Configuration
source_name = "Salesforce_NPSP"
source_id = get_source_id_by_name(source_name) or "4e5701ab-3c18-489d-93f7-0d62b78900f1"

# Feature flags
enable_create_tables = False

# Global functions

In [ ]:
def get_bronze_table(table_name: str):
    # if schemas (preview) enabled then use format {lakehouse}.dbo.{table_name}
    return spark.read.table(get_full_table_name(bronze_lakehouse_name, table_name))

# Metadata tables

## Watermark table

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, DateType, TimestampType, IntegerType, BooleanType, LongType
from pyspark.sql import SparkSession, DataFrame
import concurrent.futures
from datetime import datetime, timezone
import logging

WATERMARK_TABLE_NAME = "Metadata_Watermark"

class SalesforceNpspBronzeModel(BaseDataModel):
    def get_entities(self) -> list[tuple[StructType, str, bool]]:
        return [
            (SalesforceNpspBronzeModel.get_watermark_schema(), WATERMARK_TABLE_NAME, False)
        ]
        
    @staticmethod
    def get_watermark_schema() -> StructType:
        return StructType([
            StructField("ObjectName", StringType(), False),
            StructField("LastWatermark", TimestampType(), False),
            StructField("UpdatedAt", TimestampType(), False)
        ])


# Initialize the model and create the tables
if enable_create_tables:
    spark = SparkSession.builder.getOrCreate()

    logging.info("Creating Salesforce NPSP bronze data model entities.")

    model = SalesforceNpspBronzeModel(spark=spark, lakehouse=bronze_lakehouse_name)
    model.create_entities()

    logging.info("All tables created.")

### Transform simple dimensions function

In [ ]:
from pyspark.sql.functions import col, trim, expr, current_timestamp, lit
from pyspark.sql import DataFrame

def transform_silver_simple_dim(
    df: DataFrame,
    source_col: str,
    id_col_name: str,
    extra_cols: dict = None,
    transform_df: callable = None,
    source_id_override: str = None,
    created_col: str = None,
    modified_col: str = None,
    source_system_col: str = None
) -> DataFrame:
    """
    Universal simple-dimension enricher.

    Behavior:
    - Always generates a surrogate key column `id_col_name` with uuid().
    - If a mapped dim also uses mapping, the CDF framework can overwrite that PK later.
    - Never generates SourceSystemId; it only copies it when source_system_col is provided.
    """

    sid = source_id_override if source_id_override is not None else source_id
    ts  = current_timestamp()

    # Build select list dynamically
    select_exprs = [trim(col(source_col)).alias("Name")]

    if created_col:
        select_exprs.append(col(created_col).alias("CreatedDate"))
    if modified_col:
        select_exprs.append(col(modified_col).alias("ModifiedDate"))
    if source_system_col:
        select_exprs.append(col(source_system_col).alias("SourceSystemId"))

    base_df = (
        df
        .select(*select_exprs)
        .filter(col("Name").isNotNull() & (col("Name") != ""))
    )

    # Distinct: if we have SourceSystemId, dedupe by it, otherwise by Name
    if "SourceSystemId" in base_df.columns:
        # Filter out NULL SourceSystemId to ensure stickiness works (NULL keys cannot be joined/mapped)
        base_df = base_df.filter(col("SourceSystemId").isNotNull())
        base_df = base_df.dropDuplicates(["SourceSystemId"])
    else:
        base_df = base_df.dropDuplicates(["Name"])

    # Surrogate key
    base_df = base_df.withColumn(id_col_name, expr("uuid()"))

    # CreatedDate / ModifiedDate handling
    if "CreatedDate" not in base_df.columns:
        base_df = base_df.withColumn("CreatedDate", ts)
    if "ModifiedDate" not in base_df.columns:
        base_df = base_df.withColumn("ModifiedDate", ts)

    # Always add SourceId
    base_df = base_df.withColumn("SourceId", lit(sid))

    # Extra simple columns
    if extra_cols:
        for k, v in extra_cols.items():
            base_df = base_df.withColumn(k, v)

    # Advanced transform
    if transform_df:
        base_df = transform_df(base_df)

    return base_df
